In [1]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-01 01:05:28.501587: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-01 01:05:28.845753: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-01 01:05:29.902786: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

In [5]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=25)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty_exp", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=180, n_jobs=5)
print(study.best_trials)

[I 2026-06-01 01:10:05,589] Using an existing study with name 'diff_add_novelty_exp' instead of creating a new one.


   	                        fitness                        	                            novelty                             
   	-------------------------------------------------------	----------------------------------------------------------------
gen	avg    	gen	max     	min     	std    	avg     	gen	max     	min      	std     
0  	-338.14	0  	-87.8815	-561.809	141.658	0.242896	0  	0.897891	0.0229698	0.195207
   	                            fitness                            	                            novelty                            
   	---------------------------------------------------------------	---------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min       	std    
0  	-405.342	0  	-87.4639	-712.778	174.504	0.220417	0  	0.821986	0.00233417	0.16461
   	                            fitness                            	                            novelty                             
   	---------------

[I 2026-06-01 01:47:01,921] Trial 6 finished with value: -65.09519064088458 and parameters: {'lambda': 40, 'mutation_rate': 0.33, 'cross_rate': 0.8, 'start_fit_w': 0.7, 'decay': 4.5}. Best is trial 6 with value: -65.09519064088458.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A clas

14 	-358.373	14 	-11.5433	-678.318	177.643	0.375996	14 	0.711172	0.00163325 	0.191068
14 	-385.46 	14 	3.05532 	-672.309	180.85 	0.342853	14 	0.704644	0.00375791	0.184085
   	                            fitness                            	                            novelty                            
   	---------------------------------------------------------------	---------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min      	std    
0  	-363.603	0  	-103.371	-552.597	138.888	0.377583	0  	0.826911	0.0379985	0.24305
   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min       	std     
0  	-385.433	0  	-88.0261	-712.513	175.138	0.443

[I 2026-06-01 01:59:53,327] Trial 9 finished with value: 0.21203780355286975 and parameters: {'lambda': 50, 'mutation_rate': 0.46, 'cross_rate': 0.8, 'start_fit_w': 0.9000000000000001, 'decay': 3.0}. Best is trial 9 with value: 0.21203780355286975.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: Runti

...

1  	-396.649	1  	-12.1332	-726.101	175.252	0.302664	1  	0.648042	0.0313669	0.156513
1  	-387.278	1  	-90.6226	-712.513	170.854	0.298563	1  	0.687205	0.00483602	0.158813
2  	-338.271	2  	-54.171 	-561.185	146.86 	0.293725	2  	0.65431 	0.00337107	0.163237
6  	-367.875	6  	-15.5604	-712.513	190.656	0.301079	6  	0.674765	0.00163374	0.158615
1  	-396.649	1  	-12.1332	-726.101	175.252	0.302664	1  	0.648042	0.0313669	0.156513
6  	-404.034	6  	-30.9188	-658.757	161.696	0.278008	6  	0.601408	0.0113344  	0.138141
2  	-343.453	2  	-76.6882	-573.399	146.668	0.311121	2  	0.666775	0.0212976	0.157254
7  	-345.761	7  	-71.4527	-617.508	137.792	0.323454	7  	0.732204	0.0637853 	0.157753
2  	-404.023	2  	-100.181	-712.513	154.419	0.301114	2  	0.753916	0.00458311	0.166085
2  	-361.885	2  	2.75909 	-687.54 	153.3  	0.308932	2  	0.646955	0.0391452  	0.146422
2  	-343.453	2  	-76.6882	-573.399	146.668	0.311121	2  	0.666775	0.0212976	0.157254
2  	-401.797	2  	-74.2771	-712.513	164.376	0.29774 	2  	0.771086	0.

[I 2026-06-02 02:17:41,736] Trial 179 finished with value: 72.50437088943481 and parameters: {'lambda': 60, 'mutation_rate': 0.07, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.5}. Best is trial 85 with value: 82.85210520757455.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class n

   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min     	std     
0  	-357.539	0  	-91.1992	-596.391	134.434	0.310508	0  	0.795728	0.067223	0.167483
   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min     	std    	avg     	gen	max     	min     	std     
0  	-403.019	0  	-35.6292	-630.771	162.684	0.307212	0  	0.761093	0.010928	0.188368
   	                            fitness                            	                            novelty                             
   

[I 2026-06-02 02:32:31,054] Trial 180 finished with value: 72.50437088943481 and parameters: {'lambda': 60, 'mutation_rate': 0.07, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:32:56,164] Trial 181 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:33:08,641] Trial 182 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:33:33,282] Trial 183 finished with value: 82.85210520757455 and parameters: {'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 2.0}. Best is trial 85 with value: 82.85210520757455.
[I 2026-06-02 02:35:37,440] Trial 184 finished with value: 82.852105

[FrozenTrial(number=85, state=<TrialState.COMPLETE: 1>, values=[82.85210520757455], datetime_start=datetime.datetime(2026, 6, 1, 12, 20, 38, 681537), datetime_complete=datetime.datetime(2026, 6, 1, 13, 1, 7, 636831), params={'lambda': 60, 'mutation_rate': 0.09, 'cross_rate': 0.3, 'start_fit_w': 0.5, 'decay': 1.0}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'start_fit_w': FloatDistribution(high=1.0, log=False, low=0.2, step=0.1), 'decay': FloatDistribution(high=5.0, log=False, low=0.5, step=0.5)}, trial_id=266, value=None), FrozenTrial(number=101, state=<TrialState.COMPLETE: 1>, values=[82.85210520757455], datetime_start=datetime.datetime(2026, 6, 1, 14, 41, 47, 518944), datetime_complete=datetime.datetime(2026, 6, 1, 15, 27, 6, 438380), params={'lambd